# Reward intervals and slider matching

Does matching the slider start vary with the reward outcomes since the previous happiness rating? This matters because removing matching ratings could change which reward–mood observations enter the analysis.

I build one row per pair of successive happiness ratings, retaining the reward events between them in order. I use development participants’ plays from versions 3–5, without excluding plays or ratings within those versions.

## 1. Load the data and development split

I load the same dataset and participant assignments used in notebook 3, then select development participants before constructing the intervals.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from gbe_loader import load_gbe


participants = load_gbe()

def get_plays(participant):
    # same container handling as notebook 2
    if participant["nPlays"] == 1:
        return [participant["data"]]
    return list(participant["data"])

In [2]:
import hashlib

SPLIT_PATH = Path.cwd().parent / "splits" / "gbe_split_2026-10-07.csv"
EXPECTED_SPLIT_SHA256 = "861de3c444f506c76cbac3208840fd62d57a71b6aad63fceb83a802d8da139e9"

# check that this is the original saved split file
assert hashlib.sha256(SPLIT_PATH.read_bytes()).hexdigest() == EXPECTED_SPLIT_SHA256
split = pd.read_csv(SPLIT_PATH)

assert list(split.columns) == ["id", "set"]
assert pd.api.types.is_integer_dtype(split["id"])
assert split["id"].is_unique
assert set(split["set"]) == {"development", "holdout"}

# check that the assignments cover exactly the loaded participants
loaded_ids = [int(person["id"]) for person in participants]
assert len(loaded_ids) == len(set(loaded_ids))
assert set(split["id"]) == set(loaded_ids)

development_ids = set(split.loc[split["set"] == "development", "id"])
holdout_ids = set(split.loc[split["set"] == "holdout", "id"])
assert development_ids.isdisjoint(holdout_ids)
assert len(development_ids) == 10_000
assert len(holdout_ids) == 37_067

# leave participants unchanged; keep the released ordering in this subset
development_participants = [
    person for person in participants if int(person["id"]) in development_ids
]
assert len(development_participants) == 10_000

print("Split checksum and IDs verified.")
print("Development people:", len(development_participants))
print("Reserved hold-out IDs:", len(holdout_ids))

Split checksum and IDs verified.
Development people: 10000
Reserved hold-out IDs: 37067


## 2. Work through one rating interval

I start with the interval from baseline to the first post-outcome rating. Baseline precedes the task, and the next rating follows trial 3, so this interval contains trials 1–3.

In [3]:
for participant in development_participants:
    versions = np.atleast_1d(participant["designVersion"])

    eligible = np.flatnonzero(np.isin(versions, [3, 4, 5]))
    if len(eligible):
        play_index = int(eligible[0])
        example_participant = participant
        example_play = get_plays(participant)[play_index]
        example_version = int(versions[play_index])
        break

print("Participant:", example_participant["id"])
print("Available play:", play_index + 1)
print("Version:", example_version)

Participant: 128
Available play: 2
Version: 4


In [4]:
rating_rows = np.flatnonzero(~np.isnan(example_play[:, 9]))
boundaries = example_play[rating_rows, 0].astype(int).copy() # get the trial numbers
boundaries[0] = 0  # baseline precedes trial 1

left, right = boundaries[:2] # 0 and 3
events = example_play[left:right] # rows 0–2: trials 1–3

print("Starting happiness:", example_play[rating_rows[0], 9])
print("Ending happiness:", example_play[rating_rows[1], 9])

pd.DataFrame(
    events[:, [0, 2, 3, 4, 6, 7]],
    columns=[
        "trial", "certainValue", "winValue",
        "loseValue", "choseRisky", "outcome",
    ],
)

Starting happiness: 52.0
Ending happiness: 50.0


,trial,certainValue,winValue,loseValue,choseRisky,outcome
0,1.0,55.0,124.0,0.0,1.0,124.0
1,2.0,0.0,55.0,-37.0,1.0,55.0
2,3.0,0.0,40.0,-28.0,1.0,-28.0


### Reconstruct the reward terms

All three choices are gambles. Calculating EV and RPE by hand gives EVs of 62, 9 and 6, and RPEs of 62, 46 and −34.

In [5]:
offered_ev = (events[:, 3] + events[:, 4]) / 2
chose_gamble = events[:, 6] == 1

certain_reward = np.where(chose_gamble, 0, events[:, 2])
gamble_ev = np.where(chose_gamble, offered_ev, 0)
rpe = np.where(chose_gamble, events[:, 7] - offered_ev, 0)

np.testing.assert_allclose(gamble_ev, [62, 9, 6])
np.testing.assert_allclose(rpe, [62, 46, -34])
np.testing.assert_allclose(certain_reward, [0, 0, 0])

print("Sum of RPEs:", rpe.sum())
print("Sum of absolute RPEs:", np.abs(rpe).sum())

Sum of RPEs: 74.0
Sum of absolute RPEs: 142.0


The signed RPE sum is 74; the absolute sum is 142. The signed sum allows positive and negative surprises to cancel, while the absolute sum captures their combined magnitude.

### Store the interval

I store both happiness ratings and their slider starts, together with the interval boundaries, reward sums and ordered reward sequences.

In [6]:
start_row, end_row = rating_rows[:2]

example_interval = {
    "id": int(example_participant["id"]),
    "available_play": play_index + 1,
    "version": example_version,
    "interval": 1,
    "start_boundary": int(left),
    "end_boundary": int(right),
    "n_trials": len(events),
    "prior_happiness": example_play[start_row, 9],
    "next_happiness": example_play[end_row, 9],
    "prior_startValue": example_play[start_row, 10],
    "next_startValue": example_play[end_row, 10],
    "prior_matches_start": bool(np.isclose(
        example_play[start_row, 9],
        example_play[start_row, 10],
        rtol=0, atol=1e-10,
    )),
    "next_matches_start": bool(np.isclose(
        example_play[end_row, 9],
        example_play[end_row, 10],
        rtol=0, atol=1e-10,
    )),
    "sum_rpe": rpe.sum(),
    "sum_abs_rpe": np.abs(rpe).sum(),
    "sum_certain_reward": certain_reward.sum(),
    "sum_gamble_ev": gamble_ev.sum(),
    "trial_numbers": events[:, 0].astype(int).tolist(),
    "rpe_sequence": rpe.tolist(),
    "certain_reward_sequence": certain_reward.tolist(),
    "gamble_ev_sequence": gamble_ev.tolist(),
}

pd.DataFrame([example_interval]).T

,0
id,128
available_play,2
version,4
interval,1
start_boundary,0
end_boundary,3
n_trials,3
prior_happiness,52.0
next_happiness,50.0
prior_startValue,27.0


## 3. Build the interval table

### All intervals in the example play

I put the interval calculations into a function and apply it to the example play. I check that the intervals join correctly and cover all 30 trials once, in order.

In [7]:
# return a list of interval dictionaries for specified play, participant, and version
def build_play_intervals(play, participant_id, available_play, version):
    rating_rows = np.flatnonzero(~np.isnan(play[:, 9]))

    boundaries = play[rating_rows, 0].astype(int).copy()
    boundaries[0] = 0  # baseline comes before any trials

    intervals = []

    for index in range(len(rating_rows) - 1):
        start_row = rating_rows[index]
        end_row = rating_rows[index + 1]
        left = boundaries[index]
        right = boundaries[index + 1]
        
        events = play[left:right]

        offered_ev = (events[:, 3] + events[:, 4]) / 2
        chose_gamble = events[:, 6] == 1

        certain_reward = np.where(chose_gamble, 0, events[:, 2])
        gamble_ev = np.where(chose_gamble, offered_ev, 0)
        rpe = np.where(chose_gamble, events[:, 7] - offered_ev, 0)

        interval = {
            "id": int(participant_id),
            "available_play": available_play,
            "version": version,
            "interval": index + 1,
            "start_boundary": int(left),
            "end_boundary": int(right),
            "n_trials": len(events),
            "prior_happiness": play[start_row, 9],
            "next_happiness": play[end_row, 9],
            "prior_startValue": play[start_row, 10],
            "next_startValue": play[end_row, 10],
            "prior_matches_start": bool(np.isclose(
                play[start_row, 9],
                play[start_row, 10],
                rtol=0, atol=1e-10,
            )),
            "next_matches_start": bool(np.isclose(
                play[end_row, 9],
                play[end_row, 10],
                rtol=0, atol=1e-10,
            )),
            "sum_rpe": rpe.sum(),
            "sum_abs_rpe": np.abs(rpe).sum(),
            "sum_certain_reward": certain_reward.sum(),
            "sum_gamble_ev": gamble_ev.sum(),
            "trial_numbers": events[:, 0].astype(int).tolist(),
            "rpe_sequence": rpe.tolist(),
            "certain_reward_sequence": certain_reward.tolist(),
            "gamble_ev_sequence": gamble_ev.tolist(),
        }
        intervals.append(interval)

    return intervals

example_intervals = build_play_intervals(
    example_play,
    example_participant["id"],
    play_index + 1,
    example_version,
)

print("Intervals:", len(example_intervals))
pd.DataFrame(example_intervals).head()

Intervals: 11


,id,available_play,version,interval,start_boundary,end_boundary,n_trials,prior_happiness,next_happiness,prior_startValue,...,prior_matches_start,next_matches_start,sum_rpe,sum_abs_rpe,sum_certain_reward,sum_gamble_ev,trial_numbers,rpe_sequence,certain_reward_sequence,gamble_ev_sequence
0,128,2,4,1,0,3,3,52.0,50.0,27.0,...,False,False,74.0,142.0,0.0,77.0,"[1, 2, 3]","[62.0, 46.0, -34.0]","[0.0, 0.0, 0.0]","[62.0, 9.0, 6.0]"
1,128,2,4,2,3,6,3,50.0,52.0,33.0,...,False,False,76.5,76.5,35.0,22.5,"[4, 5, 6]","[0.0, 47.0, 29.5]","[35.0, 0.0, 0.0]","[0.0, -7.0, 29.5]"
2,128,2,4,3,6,9,3,52.0,29.0,32.0,...,False,False,-164.5,164.5,0.0,145.5,"[7, 8, 9]","[-37.0, -55.5, -72.0]","[0.0, 0.0, 0.0]","[18.0, 55.5, 72.0]"
3,128,2,4,4,9,11,2,29.0,31.0,37.0,...,False,False,24.0,74.0,0.0,64.0,"[10, 11]","[-25.0, 49.0]","[0.0, 0.0]","[15.0, 49.0]"
4,128,2,4,5,11,14,3,31.0,39.0,48.0,...,False,False,6.0,156.0,0.0,-22.0,"[12, 13, 14]","[29.5, -75.0, 51.5]","[0.0, 0.0, 0.0]","[29.5, -75.0, 23.5]"


Checking that trials 1–30 appear once and in order, and that neighbouring intervals share the same rating endpoint.

In [8]:
# flatten the trial lists, keeping their order
covered_trials = [
    trial
    for interval in example_intervals
    for trial in interval["trial_numbers"]
]

# sanity checks: 11 intervals, trials 1–30 covered once, in order
assert len(example_intervals) == 11
np.testing.assert_array_equal(covered_trials, np.arange(1, 31))

# rating endpoints join correctly
for previous, following in zip(example_intervals[:-1], example_intervals[1:]):
    assert previous["end_boundary"] == following["start_boundary"]
    assert previous["next_happiness"] == following["prior_happiness"]
    assert previous["next_startValue"] == following["prior_startValue"]

print("11 intervals; trials 1–30 covered once, in order.")
print("Rating endpoints join correctly.")

11 intervals; trials 1–30 covered once, in order.
Rating endpoints join correctly.


### Apply to development plays

I build the intervals for all development plays in versions 3–5, repeating the coverage and endpoint checks for each play.

In [9]:
interval_rows = []
plays_checked = 0

for participant in development_participants:
    versions = np.atleast_1d(participant["designVersion"])

    for play_index, play in enumerate(get_plays(participant)):
        version = int(versions[play_index])
        if version not in (3, 4, 5):
            continue

        intervals = build_play_intervals(
            play, participant["id"], play_index + 1, version
        )

        # identify the play if a check fails
        play_key = (int(participant["id"]), play_index + 1)
        assert len(intervals) == 11, play_key

        covered_trials = [
            trial
            for interval in intervals
            for trial in interval["trial_numbers"]
        ]
        np.testing.assert_array_equal(
            covered_trials, np.arange(1, 31),
            err_msg=f"Trial coverage: {play_key}",
        )

        for previous, following in zip(intervals[:-1], intervals[1:]):
            assert previous["end_boundary"] == following["start_boundary"], play_key
            assert previous["next_happiness"] == following["prior_happiness"], play_key
            assert previous["next_startValue"] == following["prior_startValue"], play_key

        interval_rows.extend(intervals)
        plays_checked += 1

development_intervals = pd.DataFrame(interval_rows)

assert not development_intervals.duplicated(
    ["id", "available_play", "interval"]
).any()

print("People:", development_intervals["id"].nunique())
print("Plays checked:", plays_checked)
print("Intervals:", len(development_intervals))

People: 6675
Plays checked: 11783
Intervals: 129613


## 4. Matching and the preceding reward interval

I check whether matching varies with recent reward exposure, allowing for prior happiness, slider starting value and rating position. This helps assess whether excluding matches would change the reward–mood observations retained.

I fit a separate model for each version, using signed and absolute RPE sums, certain-reward and gamble-EV sums, prior happiness, slider start and rating position. The models include all intervals and account for clustering by participant.

These comparisons include differences both within and between participants.

### Prepare the model data

I code matching as 0/1 and divide reward sums by 100, so their coefficients describe a 100-point increase.

In [10]:
# separate modelling copy
selection_data = development_intervals.copy()

selection_data["matches_next"] = (
    selection_data["next_matches_start"].astype(int)
)

reward_fields = [
    "sum_rpe", "sum_abs_rpe",
    "sum_certain_reward", "sum_gamble_ev",
]

# add per-100 versions of the reward fields for easier interpretation
for field in reward_fields:
    selection_data[f"{field}_per100"] = selection_data[field] / 100

selection_data["last_rpe_per100"] = (
    # take the final RPE from each interval sequence and divide by 100
    selection_data["rpe_sequence"].map(lambda values: values[-1]) / 100
)
selection_data["last_abs_rpe_per100"] = (
    selection_data["last_rpe_per100"].abs()
)

print("Intervals:", len(selection_data))
print("People:", selection_data["id"].nunique())

Intervals: 129613
People: 6675


### Build the predictor matrix

I allow prior happiness and slider start to have curved relationships with matching, rather than assuming a straight-line relationship on the log-odds scale. Each is represented by a natural cubic spline with four degrees of freedom.

In [11]:
from patsy import dmatrices

selection_v3 = selection_data[
    selection_data["version"] == 3
].copy()

# predictors for whether the next rating matches its slider start
# allow curved relationships for prior happiness and the next slider start;
# 4 df gives each curve moderate flexibility using 4 predictor columns
# centre the spline bases so they can be included alongside the intercept
# treat interval position as categorical
selection_formula = """
matches_next ~
    sum_rpe_per100
    + sum_abs_rpe_per100
    + sum_certain_reward_per100
    + sum_gamble_ev_per100
    + cr(prior_happiness, df=4, constraints='center')
    + cr(next_startValue, df=4, constraints='center')
    + C(interval)
"""

y_selection, X_selection = dmatrices(
    selection_formula,
    data=selection_v3,
    return_type="dataframe",
    NA_action="raise", # if missing value stop, don't drop
)

assert len(X_selection) == len(selection_v3)
assert np.isfinite(X_selection.to_numpy()).all()

# count linearly independent predictor cols
rank = np.linalg.matrix_rank(X_selection.to_numpy())

print("Intervals:", len(X_selection))
print("Predictor columns:", X_selection.shape[1])
print("Matrix rank:", rank)

Intervals: 12804
Predictor columns: 23
Matrix rank: 23


### Fit version 3 first

I first fit the logistic regression for version 3, clustering uncertainty by participant. This model includes no play fixed effects, so its associations combine variation within and between plays.

In [12]:
import statsmodels.api as sm

selection_fit_v3 = sm.GLM(
    y_selection.iloc[:, 0],
    X_selection, # predictors
    family=sm.families.Binomial(), # binary outcome
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": selection_v3["id"],
        "use_correction": True, # use standard small-sample correction
    },
)

print("Converged:", selection_fit_v3.converged)

Converged: True


### Reward associations with matching

I show the reward coefficients as odds ratios per 100-point increase, adjusted for the other predictors. Values below 1 indicate lower odds of matching; values above 1 indicate higher odds.

Since signed and absolute RPE sums enter the model together, the signed RPE coefficient describes a shift towards more positive RPEs at the same total absolute RPE. Prior happiness, slider start and rating position are included in the model but omitted from this table.

In [13]:
reward_terms = [f"{field}_per100" for field in reward_fields]
confidence_intervals = selection_fit_v3.conf_int()

reward_results_v3 = pd.DataFrame({
    "term": reward_terms,
    "odds_ratio": np.exp(selection_fit_v3.params.loc[reward_terms]), # convert log-odds coefficients and intervals to odds ratios
    "CI_low_95": np.exp(confidence_intervals.loc[reward_terms, 0]),
    "CI_high_95": np.exp(confidence_intervals.loc[reward_terms, 1]),
})

reward_results_v3.style.format({
    "odds_ratio": "{:.3f}",
    "CI_low_95": "{:.3f}",
    "CI_high_95": "{:.3f}",
}).hide(axis="index")

term,odds_ratio,CI_low_95,CI_high_95
sum_rpe_per100,0.980,0.919,1.045
sum_abs_rpe_per100,0.784,0.687,0.895
sum_certain_reward_per100,1.172,1.055,1.302
sum_gamble_ev_per100,0.985,0.907,1.071


In version 3, greater absolute RPE exposure is associated with lower matching odds. More positive certain-reward sums are associated with higher matching odds. The confidence intervals for signed RPE and gamble EV include 1.

### Repeat across versions 3–5

I use the same formula and fitting method for each version. Spline knots are placed at quantiles of that version’s prior-happiness and slider-start values, basically placing them closer together where observations are more concentrated. I refit version 3 to check that it reproduces the worked example above.

In [14]:
selection_fits = {}
reward_result_rows = []

for version in (3, 4, 5):
    data = selection_data[
        selection_data["version"] == version
    ].copy()

    y, X = dmatrices(
        selection_formula,
        data=data,
        return_type="dataframe",
        NA_action="raise",
    )

    assert len(X) == len(data)
    assert np.linalg.matrix_rank(X.to_numpy()) == X.shape[1]

    fit = sm.GLM(
        y.iloc[:, 0], X,
        family=sm.families.Binomial(),
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": data["id"],
            "use_correction": True,
        },
    )

    assert fit.converged, f"Version {version} did not converge"
    selection_fits[version] = fit
    ci = fit.conf_int()

    for term in reward_terms:
        reward_result_rows.append({
            "version": version,
            "term": term,
            "odds_ratio": np.exp(fit.params[term]),
            "CI_low_95": np.exp(ci.loc[term, 0]),
            "CI_high_95": np.exp(ci.loc[term, 1]),
        })

reward_results = pd.DataFrame(reward_result_rows)

np.testing.assert_allclose(
    selection_fits[3].params,
    selection_fit_v3.params,
    rtol=0, atol=1e-8,
)

reward_results.style.format({
    "odds_ratio": "{:.3f}",
    "CI_low_95": "{:.3f}",
    "CI_high_95": "{:.3f}",
}).hide(axis="index")

version,term,odds_ratio,CI_low_95,CI_high_95
3,sum_rpe_per100,0.980,0.919,1.045
3,sum_abs_rpe_per100,0.784,0.687,0.895
3,sum_certain_reward_per100,1.172,1.055,1.302
3,sum_gamble_ev_per100,0.985,0.907,1.071
4,sum_rpe_per100,0.978,0.929,1.030
4,sum_abs_rpe_per100,0.832,0.738,0.938
4,sum_certain_reward_per100,0.986,0.913,1.064
4,sum_gamble_ev_per100,0.976,0.914,1.042
5,sum_rpe_per100,0.957,0.933,0.981
5,sum_abs_rpe_per100,0.794,0.745,0.846


Across versions 3–5, greater absolute RPE exposure is associated with lower matching odds after accounting for the other predictors. Excluding matching ratings would therefore change the mix of reward histories retained.

## 5. Final-trial comparison

I check whether the association with matching also appears for the most recent trial’s RPE. I replace the signed and absolute RPE sums with the final trial’s signed and absolute RPE, keeping the other predictors unchanged. All intervals remain included, with final-trial RPE coded as zero for certain choices.

In [15]:
recency_formula = selection_formula.replace(
    "sum_rpe_per100", "last_rpe_per100"
).replace(
    "sum_abs_rpe_per100", "last_abs_rpe_per100"
)

recency_terms = [
    "last_rpe_per100",
    "last_abs_rpe_per100",
    "sum_certain_reward_per100",
    "sum_gamble_ev_per100",
]

In [16]:
recency_fits = {}
recency_result_rows = []

for version in (3, 4, 5):
    data = selection_data[
        selection_data["version"] == version
    ].copy()

    y, X = dmatrices(
        recency_formula,
        data=data,
        return_type="dataframe",
        NA_action="raise",
    )

    assert len(X) == len(data)
    assert np.linalg.matrix_rank(X.to_numpy()) == X.shape[1]

    fit = sm.GLM(
        y.iloc[:, 0], X,
        family=sm.families.Binomial(),
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": data["id"],
            "use_correction": True,
        },
    )

    assert fit.converged, f"Version {version} did not converge"
    recency_fits[version] = fit
    ci = fit.conf_int()
    
    for term in recency_terms:
        recency_result_rows.append({
            "version": version,
            "term": term,
            "odds_ratio": np.exp(fit.params[term]),
            "CI_low_95": np.exp(ci.loc[term, 0]),
            "CI_high_95": np.exp(ci.loc[term, 1]),
        })

recency_results = pd.DataFrame(recency_result_rows)

recency_results.style.format({
    "odds_ratio": "{:.3f}",
    "CI_low_95": "{:.3f}",
    "CI_high_95": "{:.3f}",
}).hide(axis="index")

version,term,odds_ratio,CI_low_95,CI_high_95
3,last_rpe_per100,0.950,0.861,1.050
3,last_abs_rpe_per100,0.734,0.616,0.875
3,sum_certain_reward_per100,1.153,1.038,1.279
3,sum_gamble_ev_per100,0.970,0.895,1.050
4,last_rpe_per100,0.932,0.862,1.007
4,last_abs_rpe_per100,0.698,0.591,0.825
4,sum_certain_reward_per100,0.975,0.901,1.054
4,sum_gamble_ev_per100,0.970,0.914,1.030
5,last_rpe_per100,0.926,0.888,0.966
5,last_abs_rpe_per100,0.723,0.663,0.788


Greater absolute RPE on the final trial is associated with lower matching odds in all three versions, after accounting for the other predictors. The pattern therefore also appears when using the most recent trial’s RPE instead of the interval sum.

## 6. Conclusion

The interval table covers all 30 trials once per play and retains the ordered reward sequences alongside their summaries.

Across versions 3–5, matching is less likely after greater absolute RPE exposure, using either interval sums or the final trial’s RPE. Excluding matching ratings would therefore change the reward histories represented in the analysis.

I retain matching ratings for now. Next, I implement the published happiness model before testing the provisional mood × RPE model on synthetic data.